# Calibration, Counterfactual Fairness, and Proxy Risk Lab


## Purpose

This lab introduces calibration diagnostics and proxy-risk thinking.

Learning goals:

- Build calibration bins by group.
- Simulate proxy variables.
- Connect statistical audit to causal questions.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 2500

frame = pd.DataFrame({
    "group": rng.choice(["A", "B"], size=n, p=[0.55, 0.45]),
})

frame["proxy_variable"] = np.where(
    frame["group"] == "A",
    rng.normal(0.25, 1.0, size=n),
    rng.normal(-0.25, 1.0, size=n),
)

raw_score = 1 / (1 + np.exp(-(0.8 * frame["proxy_variable"] + rng.normal(0, 1, size=n))))
frame["score"] = raw_score
frame["target"] = rng.binomial(1, frame["score"])

frame["confidence_bin"] = pd.cut(
    frame["score"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True,
)

calibration = (
    frame
    .groupby(["group", "confidence_bin"], observed=True)
    .agg(
        n=("target", "size"),
        mean_confidence=("score", "mean"),
        empirical_rate=("target", "mean"),
    )
    .reset_index()
)

proxy_summary = (
    frame
    .groupby("group", as_index=False)
    .agg(
        mean_proxy=("proxy_variable", "mean"),
        mean_score=("score", "mean"),
        base_rate=("target", "mean"),
    )
)

calibration.head(), proxy_summary

## Interpretation

Calibration tells whether scores behave like probabilities. Proxy analysis asks whether variables transmit protected-attribute structure through the model.